# Notebook 9: Dashboard — página única para audiencia no técnica

Junta todo lo que ya se generó en una sola página HTML autocontenida:

- Los 3 mapas interactivos (notebooks 06 y 07) — embebidos como iframes.
- Clima × severidad (notebook 03) — gráfico interactivo.
- Tendencia nacional 2000-2024 (notebook 05) — gráfico interactivo.
- Resultados del modelo de riesgo (notebook 08) — **no reentrena nada**, reusa el CSV que ya
  guardó `08_modelo_riesgo_comuna_anio.ipynb` con las predicciones (`riesgo` real vs
  `riesgo_predicho`).

**Importante:** el `dashboard.html` final debe quedar en la **misma carpeta** que los 3
mapas (`mapa_severidad_historica.html`, `mapa_estres_hidrico.html`,
`mapa_puntos_importancia.html`) — los iframes los referencian por ruta relativa. Si luego
suben esto a GitHub Pages, suban los 4 HTML juntos, en la misma carpeta.


## 1. Montar Google Drive

In [12]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Instalar librerías

In [13]:
!pip install -q plotly
print('✓ Librerías instaladas')


✓ Librerías instaladas


## 3. Configuración — EDITA la ruta antes de correr

In [16]:
import os, glob

# ⚠️ Debe ser la MISMA carpeta base usada en los notebooks anteriores
DRIVE_BASE = '/content/drive/MyDrive/BigDataSIC'
OUTPUT_DIR = f'{DRIVE_BASE}/datos_procesados'

PATH_EVENTOS   = f'{OUTPUT_DIR}/incendios_conaf_era5_2010_2020.csv'          # notebook 03
PATH_NACIONAL  = f'{OUTPUT_DIR}/incendios_nacional_mensual_era5.csv'        # notebook 05
PATH_RIESGO    = f'{OUTPUT_DIR}/modelo_riesgo_comuna_anio.csv'              # notebook 08
SPARK_CSV_GLOB = f'{OUTPUT_DIR}/spark_comuna_anio/csv/part-*.csv'           # notebook 04

DASHBOARD_OUT = f'{OUTPUT_DIR}/dashboard.html'

print('Todas las rutas apuntan a:', OUTPUT_DIR)


Todas las rutas apuntan a: /content/drive/MyDrive/BigDataSIC/datos_procesados


## 4. Cargar los datasets ya generados (sin recalcular nada)

In [17]:
import pandas as pd
import numpy as np

eventos = pd.read_csv(PATH_EVENTOS)
nacional = pd.read_csv(PATH_NACIONAL)
riesgo = pd.read_csv(PATH_RIESGO)

spark_csv = glob.glob(SPARK_CSV_GLOB)
assert spark_csv, f'No se encontró {SPARK_CSV_GLOB} — ¿corriste el notebook 04?'
comuna_anio = pd.read_csv(spark_csv[0])

print(f'✓ eventos      : {len(eventos)} filas')
print(f'✓ nacional     : {len(nacional)} filas')
print(f'✓ riesgo       : {len(riesgo)} filas')
print(f'✓ comuna_anio  : {len(comuna_anio)} filas')


✓ eventos      : 60530 filas
✓ nacional     : 294 filas
✓ riesgo       : 578 filas
✓ comuna_anio  : 2617 filas


## 5. KPIs generales

In [18]:
kpis = {
    'Eventos analizados (2010-2019)': f'{len(eventos):,}',
    'Superficie quemada total': f"{eventos['superficie_ha'].sum():,.0f} ha",
    'Comunas con al menos 1 evento': f"{eventos['comuna'].nunique()}",
    'Regiones cubiertas': f"{eventos['region'].nunique()}",
    'Período serie nacional (clima real)': f"{int(nacional['anio'].min())}-{int(nacional['anio'].max())}",
}
for k, v in kpis.items():
    print(f'{k}: {v}')


Eventos analizados (2010-2019): 60,530
Superficie quemada total: 1,157,990 ha
Comunas con al menos 1 evento: 312
Regiones cubiertas: 16
Período serie nacional (clima real): 2000-2024


## 6. Gráfico — clima × severidad (muestra de eventos)

Temperatura del día en el eje X, severidad en el eje Y, y **estrés hídrico combinado en
color** — el mismo índice que usan los mapas de comuna (`z(anomalía de temperatura) -
z(anomalía de precipitación)`, calculado acá a nivel de evento individual, no comuna-año).
Escala divergente azul→amarillo pálido→rojo: rojo = mes/celda anómalamente caliente Y seco a
la vez, azul = anómalamente fresco y húmedo, amarillo = normal. Se usa **percentil relativo**
(no el valor crudo del índice) por la misma razón que los mapas: 1-2 eventos extremos pueden
estirar la escala y aplastar la variación real del resto.

Incluye un **filtro por región** (menú desplegable arriba a la derecha del gráfico) — todas
las regiones comparten la misma escala de color.


In [19]:
import plotly.graph_objects as go
import numpy as np

# Estrés hídrico a nivel de EVENTO (mismo índice que estres_hidrico_score de los mapas,
# aplicado acá por evento individual): z-score de la anomalía de temperatura menos z-score
# de la anomalía de precipitación. Se calcula sobre TODOS los eventos, antes de muestrear.
z_t2m = (eventos['t2m_anomaly_mensual'] - eventos['t2m_anomaly_mensual'].mean()) / eventos['t2m_anomaly_mensual'].std()
z_tp = (eventos['tp_anomaly_mensual'] - eventos['tp_anomaly_mensual'].mean()) / eventos['tp_anomaly_mensual'].std()
eventos['estres_hidrico_evento'] = z_t2m - z_tp

referencia_estres = np.sort(eventos['estres_hidrico_evento'].dropna().values)

def a_percentil_estres(v):
    return 100 * np.searchsorted(referencia_estres, v, side='right') / len(referencia_estres)

muestra = eventos.sample(n=min(5000, len(eventos)), random_state=42).copy()
muestra['estres_percentil'] = muestra['estres_hidrico_evento'].apply(a_percentil_estres)
regiones_disponibles = sorted(muestra['region'].dropna().unique())

fig_clima = go.Figure()
for reg in regiones_disponibles:
    sub = muestra[muestra['region'] == reg]
    fig_clima.add_trace(go.Scatter(
        x=sub['t2m_c'], y=sub['log_superficie_ha'],
        mode='markers',
        marker=dict(color=sub['estres_percentil'], coloraxis='coloraxis', size=6, opacity=0.6),
        name=reg,
        customdata=np.stack([sub['comuna'], sub['tp_mm'], sub['estres_hidrico_evento']], axis=-1),
        hovertemplate=(
            'Comuna: %{customdata[0]}<br>Temp: %{x:.1f} °C<br>'
            'Severidad (log): %{y:.2f}<br>Precipitación: %{customdata[1]:.1f} mm<br>'
            'Estrés hídrico (score): %{customdata[2]:+.2f}<extra></extra>'
        ),
    ))

n_traces = len(regiones_disponibles)
botones = [dict(label='Todas las regiones', method='update', args=[{'visible': [True] * n_traces}])]
for i, reg in enumerate(regiones_disponibles):
    visibilidad = [False] * n_traces
    visibilidad[i] = True
    botones.append(dict(label=reg, method='update', args=[{'visible': visibilidad}]))

fig_clima.update_layout(
    title=f'Clima del día del evento vs severidad (muestra de {len(muestra):,} de {len(eventos):,} eventos)',
    xaxis_title='Temperatura del día (°C)',
    yaxis_title='log(1 + superficie_ha)',
    coloraxis=dict(
        colorscale=[[0, '#2166ac'], [0.5, '#ffffbf'], [1, '#b2182b']], cmin=0, cmax=100,
        colorbar=dict(title='Estrés hídrico<br>(percentil)', ticksuffix='%'),
    ),
    template='plotly_white', height=460, showlegend=False,
    updatemenus=[dict(buttons=botones, direction='down', x=1.18, y=1.15, showactive=True)],
)
fig_clima.show()


## 7. Gráfico — tendencia nacional (clima real, 2000-2024)

In [20]:
anual = nacional.groupby('anio').agg(
    n_incendios=('n_incendios', 'sum'),
    t2m_anomaly_nacional=('t2m_anomaly_nacional', 'mean'),
    tp_anomaly_nacional=('tp_anomaly_nacional', 'mean'),
).reset_index()

fig_nacional = go.Figure()
fig_nacional.add_trace(go.Bar(x=anual['anio'], y=anual['n_incendios'], name='Incendios/año', marker_color='tomato'))
fig_nacional.add_trace(go.Scatter(
    x=anual['anio'], y=anual['t2m_anomaly_nacional'] * 200, name='Anomalía temp. (escalada ×200 para visual)',
    yaxis='y2', line=dict(color='black', width=2),
))
fig_nacional.update_layout(
    title='Incendios por año vs anomalía de temperatura nacional (ERA5, 2000-2024)',
    template='plotly_white', height=420,
    yaxis=dict(title='N° incendios/año'),
    yaxis2=dict(title='Anomalía temp. (escalada)', overlaying='y', side='right', showgrid=False),
)
fig_nacional.show()


## 8. Gráfico — resultados del modelo de riesgo (notebook 08, sin reentrenar)

In [21]:
import plotly.express as px

CLASES = ['bajo', 'normal', 'alto', 'catastrofico']

accuracy = (riesgo['riesgo'] == riesgo['riesgo_predicho']).mean()
print(f'Accuracy (recalculado desde el CSV guardado): {accuracy:.3f}')

matriz = pd.crosstab(riesgo['riesgo'], riesgo['riesgo_predicho']).reindex(index=CLASES, columns=CLASES, fill_value=0)

fig_matriz = px.imshow(
    matriz, text_auto=True, color_continuous_scale='Blues',
    labels=dict(x='Predicho', y='Real', color='N° comuna-años'),
    title=f'Matriz de confusión — modelo de riesgo (accuracy={accuracy:.2f})',
)
fig_matriz.update_layout(template='plotly_white', height=420)
fig_matriz.show()

ultimo_anio = riesgo['año'].max()
top_riesgo = (
    riesgo[riesgo['año'] == ultimo_anio]
    .sort_values('prob_catastrofico', ascending=False)
    .head(15)
)
fig_top = px.bar(
    top_riesgo, x='prob_catastrofico', y='comuna', color='region', orientation='h',
    title=f'Top 15 comunas por probabilidad de riesgo catastrófico — {int(ultimo_anio)}',
    labels={'prob_catastrofico': 'Probabilidad estimada', 'comuna': ''},
)
fig_top.update_layout(template='plotly_white', height=500, yaxis=dict(autorange='reversed'))
fig_top.show()


Accuracy (recalculado desde el CSV guardado): 0.362


## 9. Ensamblar el dashboard (una sola página HTML)

Estructura: KPIs arriba, 3 mapas en pestañas (severidad histórica / estrés hídrico / puntos
de importancia), y los 3 gráficos interactivos abajo. Los mapas se embeben como `iframe`
apuntando a los archivos `.html` que ya están en la misma carpeta — no se duplican.


In [22]:
from plotly.offline import plot as plotly_div

def fig_a_div(fig):
    return plotly_div(fig, include_plotlyjs=False, output_type='div')

div_clima = fig_a_div(fig_clima)
div_nacional = fig_a_div(fig_nacional)
div_matriz = fig_a_div(fig_matriz)
div_top = fig_a_div(fig_top)

kpi_cards = ''.join(
    f'<div class="kpi"><div class="kpi-valor">{v}</div><div class="kpi-etiqueta">{k}</div></div>'
    for k, v in kpis.items()
)

html = f'''<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<title>Incendios Forestales Chile × Clima — Dashboard</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
  body {{ font-family: -apple-system, Arial, sans-serif; margin: 0; background: #f7f7f5; color: #1a1a1a; }}
  header {{ background: #1a1a2e; color: white; padding: 24px 32px; }}
  header h1 {{ margin: 0 0 6px 0; font-size: 22px; }}
  header p {{ margin: 0; color: #c8c8d8; font-size: 13px; }}
  section {{ padding: 24px 32px; }}
  h2 {{ font-size: 16px; border-bottom: 2px solid #ddd; padding-bottom: 6px; }}
  .explicacion {{ font-size: 13px; color: #555; max-width: 780px; margin: 6px 0 16px 0; line-height: 1.5; }}
  .kpis {{ display: flex; flex-wrap: wrap; gap: 14px; }}
  .kpi {{ background: white; border-radius: 10px; padding: 14px 20px; box-shadow: 0 1px 4px rgba(0,0,0,0.08); min-width: 160px; }}
  .kpi-valor {{ font-size: 22px; font-weight: 700; color: #d1495b; }}
  .kpi-etiqueta {{ font-size: 12px; color: #666; margin-top: 4px; }}
  .tabs {{ display: flex; gap: 8px; margin-bottom: 10px; }}
  .tab-btn {{ padding: 8px 16px; border: none; border-radius: 6px 6px 0 0; background: #ddd; cursor: pointer; font-size: 13px; }}
  .tab-btn.activo {{ background: #1a1a2e; color: white; }}
  .mapa-frame {{ display: none; }}
  .mapa-frame.activo {{ display: block; }}
  iframe {{ width: 100%; height: 620px; border: 1px solid #ddd; border-radius: 6px; }}
  .grid2 {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }}
  @media (max-width: 900px) {{ .grid2 {{ grid-template-columns: 1fr; }} }}
</style>
</head>
<body>

<header>
  <h1>🔥 Incendios Forestales Chile × Clima</h1>
  <p>Samsung Innovation Campus — Big Data 2026 · Cruce ERA5 (2000-2025) × CONAF (2010-2019) ·
     detección de estrés hídrico y severidad a nivel de comuna</p>
</header>

<section>
  <h2>Resumen</h2>
  <div class="kpis">{kpi_cards}</div>
</section>

<section>
  <h2>Mapas</h2>
  <div class="tabs">
    <button class="tab-btn activo" onclick="mostrarMapa(0, this)">Severidad histórica</button>
    <button class="tab-btn" onclick="mostrarMapa(1, this)">Estrés hídrico</button>
    <button class="tab-btn" onclick="mostrarMapa(2, this)">Puntos de importancia</button>
  </div>
  <div class="mapa-frame activo"><iframe src="mapa_severidad_historica.html"></iframe></div>
  <div class="mapa-frame"><iframe src="mapa_estres_hidrico.html"></iframe></div>
  <div class="mapa-frame"><iframe src="mapa_puntos_importancia.html"></iframe></div>
</section>

<section>
  <h2>Clima y severidad</h2>
  <p class="explicacion">
    Cada punto es un incendio individual. Eje X: temperatura del día del evento. Eje Y:
    severidad (superficie quemada, escala logarítmica para que los incendios más grandes no
    aplasten visualmente al resto). Color: <b>estrés hídrico combinado</b> — el mismo índice
    de los mapas de comuna (temperatura anómalamente alta + precipitación anómalamente baja a
    la vez), calculado acá por evento individual. Rojo = caliente y seco a la vez respecto a
    lo normal de esa celda/mes, azul = fresco y húmedo, amarillo = normal. Se muestra en
    percentil relativo (no el valor crudo) para que 1-2 eventos extremos no aplasten la
    variación del resto — el score exacto y los mm de lluvia aparecen al pasar el cursor. Si
    la hipótesis del proyecto se cumple, debería verse más rojo hacia arriba del gráfico
    (incendios más severos) que hacia abajo.
  </p>
  {div_clima}
</section>

<section>
  <h2>Tendencia nacional (2000-2024)</h2>
  <p class="explicacion">
    Barras: total de incendios por año a nivel país. Línea: anomalía de temperatura nacional
    de ERA5 (promedio real de todo Chile, no un solo punto) — reescalada solo para que se vea
    en el mismo gráfico, no representa grados directos. Sirve para ver si los años con más
    incendios coinciden con años anómalamente cálidos a nivel nacional.
  </p>
  {div_nacional}
</section>

<section>
  <h2>Modelo de riesgo por comuna</h2>
  <p class="explicacion">
    Izquierda: matriz de confusión del modelo de riesgo (notebook 08) — filas son la
    categoría real, columnas la predicha; la diagonal son los aciertos. Derecha: las 15
    comunas con mayor probabilidad predicha de riesgo catastrófico en el último año
    disponible del set de prueba, coloreadas por región.
  </p>
  <div class="grid2">
    <div>{div_matriz}</div>
    <div>{div_top}</div>
  </div>
</section>

<script>
function mostrarMapa(idx, boton) {{
  document.querySelectorAll('.mapa-frame').forEach((el, i) => el.classList.toggle('activo', i === idx));
  document.querySelectorAll('.tab-btn').forEach(b => b.classList.remove('activo'));
  boton.classList.add('activo');
}}
</script>

</body>
</html>
'''

with open(DASHBOARD_OUT, 'w', encoding='utf-8') as f:
    f.write(html)

print(f'✓ Dashboard guardado: {DASHBOARD_OUT}')
print(f'  Tamaño: {os.path.getsize(DASHBOARD_OUT) / 1024:.0f} KB')


✓ Dashboard guardado: /content/drive/MyDrive/BigDataSIC/datos_procesados/dashboard.html
  Tamaño: 476 KB


## ✅ Checklist de validación

- [x] KPIs, gráficos y mapas provienen de archivos ya generados por los notebooks 03-08 —
      este notebook no reentrena ni recalcula el modelo, solo lee y visualiza.
- [x] Los mapas se embeben por `iframe` con ruta relativa — **al mover/subir el proyecto,
      los 4 archivos HTML (`dashboard.html` + los 3 mapas) deben quedar en la misma carpeta**.
- [x] El scatter de clima×severidad usa una muestra de 5.000 eventos (de 60.530) para que el
      HTML no pese demasiado — la correlación real se calculó sobre el dataset completo en
      el notebook 03.

**Para abrir el dashboard:** doble clic en `dashboard.html` desde Drive, o descargar los 4
archivos HTML juntos y abrir `dashboard.html` en cualquier navegador.
